In [2]:
import pandas as pd
import numpy as np

df_train=pd.read_csv("train (1).csv")
df_test=pd.read_csv("test (1).csv")

df_train.head()

,id,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
0,0,785.0,NaN,24472.0,F,NaN,NaN,NaN,N,0.8,NaN,3.01,NaN,NaN,NaN,NaN,80.0,11.1,4.0,D
1,1,1639.0,NaN,18628.0,F,NaN,NaN,NaN,N,0.8,NaN,3.33,NaN,NaN,NaN,NaN,273.0,10.4,3.0,C
2,2,1095.0,NaN,21185.0,F,NaN,NaN,NaN,N,0.6,NaN,3.76,NaN,NaN,NaN,NaN,312.0,10.0,4.0,C
3,3,1581.0,NaN,24472.0,F,NaN,NaN,NaN,N,1.0,NaN,3.48,NaN,NaN,NaN,NaN,277.0,10.0,2.0,C
4,4,3222.0,Placebo,18713.0,F,N,Y,Y,Y,2.5,408.0,3.70,145.0,856.0,110.05,98.0,132.0,11.0,4.0,D


In [3]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             15000 non-null  int64  
 1   N_Days         15000 non-null  float64
 2   Drug           8365 non-null   str    
 3   Age            15000 non-null  float64
 4   Sex            15000 non-null  str    
 5   Ascites        8370 non-null   str    
 6   Hepatomegaly   8361 non-null   str    
 7   Spiders        8355 non-null   str    
 8   Edema          15000 non-null  str    
 9   Bilirubin      15000 non-null  float64
 10  Cholesterol    6510 non-null   float64
 11  Albumin        15000 non-null  float64
 12  Copper         8253 non-null   float64
 13  Alk_Phos       8350 non-null   float64
 14  SGOT           8350 non-null   float64
 15  Tryglicerides  6463 non-null   float64
 16  Platelets      14408 non-null  float64
 17  Prothrombin    14980 non-null  float64
 18  Stage          15

In [4]:
df_train.isna().sum()

id                  0
N_Days              0
Drug             6635
Age                 0
Sex                 0
Ascites          6630
Hepatomegaly     6639
Spiders          6645
Edema               0
Bilirubin           0
Cholesterol      8490
Albumin             0
Copper           6747
Alk_Phos         6650
SGOT             6650
Tryglicerides    8537
Platelets         592
Prothrombin        20
Stage               0
Status              0
dtype: int64

In [ ]:
cat_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Stage']

for col in cat_cols:
    df_train[col] = df_train[col].fillna('Missing').astype(str)
    df_test[col] = df_test[col].fillna('Missing').astype(str)

In [14]:
import pandas as pd
from catboost import CatBoostClassifier

# 1. Правильный маппинг целевой переменной ('C', 'CL', 'D')
target_map = {'C': 0, 'CL': 1, 'D': 2}
y = df_train['Status'].map(target_map)

# 2. Создаем копии X и X_test
X = df_train.drop(columns=['id', 'Status']).copy()
X_test = df_test.drop(columns=['id']).copy()

# 3. Фича-инжиниринг: количество пропусков
X['null_count'] = X.isna().sum(axis=1)
X_test['null_count'] = X_test.isna().sum(axis=1)

# 4. Список категориальных колонок
cat_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Stage']

# Приведение к строке
for col in cat_cols:
    X[col] = X[col].fillna('Missing').astype(str)
    X_test[col] = X_test[col].fillna('Missing').astype(str)

# 5. Обучение модели
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    cat_features=cat_cols,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    random_seed=42,
    verbose=100
)

model.fit(X, y)

# 6. Предсказание вероятностей и сохранение сабмита
preds_proba = model.predict_proba(X_test)

submission = pd.DataFrame({
    'id': df_test['id'],
    'Status_C': preds_proba[:, 0],   # Класс 0 ('C')
    'Status_CL': preds_proba[:, 1],  # Класс 1 ('CL')
    'Status_D': preds_proba[:, 2]    # Класс 2 ('D')
})

submission.to_csv('submission.csv', index=False)
print("Ура! Все отработало без ошибок, submission.csv создан.")

0:	learn: 1.0624731	total: 275ms	remaining: 4m 34s
100:	learn: 0.4157729	total: 12.3s	remaining: 1m 49s
200:	learn: 0.3806385	total: 24.3s	remaining: 1m 36s
300:	learn: 0.3667522	total: 41s	remaining: 1m 35s
400:	learn: 0.3530448	total: 59.5s	remaining: 1m 28s
500:	learn: 0.3427560	total: 1m 21s	remaining: 1m 20s
600:	learn: 0.3336470	total: 1m 42s	remaining: 1m 8s
700:	learn: 0.3262014	total: 1m 55s	remaining: 49.3s
800:	learn: 0.3189721	total: 2m 7s	remaining: 31.6s
900:	learn: 0.3127744	total: 2m 18s	remaining: 15.2s
999:	learn: 0.3072153	total: 2m 31s	remaining: 0us
Ура! Все отработало без ошибок, submission.csv создан.
